In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyspark

## 1. Configuración

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType

spark = (SparkSession.builder
         .appName("RetoCAETEC")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())

RUTA_HISTORICO = "/content/drive/My Drive/Datos/Crudo/Datos vacas historico hasta enero 2026/TEC-UAQ-UNAM"
RUTA_PROCESADO = "/content/drive/My Drive/Datos/Procesado"
os.makedirs(RUTA_PROCESADO, exist_ok=True)

UMBRAL_COLA_MIN = 2

print("¿Existe la carpeta de datos?", os.path.exists(RUTA_HISTORICO))


## 2. Extracción


In [ ]:
produccion = [
    "hora_inicio", "numero_ordenio", "duracion", "produccion_kg", "rcs_celulas_ml", "intervalo_ordenio",
    "produccion_di", "produccion_dd", "produccion_ti", "produccion_td",
    "estado_md_ubre", "estado_md_pezon",
    "media_flujo_di", "media_flujo_dd", "media_flujo_ti", "media_flujo_td",
    "flujo_maximo_di", "flujo_maximo_dd", "flujo_maximo_ti", "flujo_maximo_td",
    "sangre_di", "sangre_dd", "sangre_ti", "sangre_td",
    "conductividad_di", "conductividad_dd", "conductividad_ti", "conductividad_td",
    "contador_ubre", "contador_di", "contador_dd", "contador_td", "contador_ti",
    "destino_leche", "razon_desviacion", "programa_lavado", "ms", "usuario",
    "patada", "incompleto", "pezones_no_encotnrados", "mdi",
]

archivos = [f for f in os.listdir(RUTA_HISTORICO)
            if f.lower().endswith((".xlsx", ".xls")) and not f.lower().startswith("concentrado consumido")]

partes = []
for archivo in archivos:
    try:
        df_temp = pd.read_excel(os.path.join(RUTA_HISTORICO, archivo), skiprows=2, names=produccion)
    except Exception as e:
        print(f"Error en {archivo}: {e}")
        continue
    df_temp["id_vaca"] = "".join(filter(str.isdigit, archivo)) or "desconocido"
    df_temp["origen_archivo"] = archivo
    partes.append(df_temp)

pdf = pd.concat(partes, ignore_index=True)
print(f"Archivos leídos: {len(partes)} | Registros: {len(pdf):,} | Columnas: {pdf.shape[1]}")

# Todo como texto para que Spark no falle con columnas de tipos mezclados;
# los tipos correctos se asignan en la limpieza.
pdf = pdf.astype(str).where(pdf.notna(), None)
esquema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
sdf = spark.createDataFrame(pdf, schema=esquema)
del pdf, partes

print(f"Registros en Spark: {sdf.count():,}")

## 3. Limpieza
| Regla | Motivo (Data Understanding) |
|---|---|
| Eliminar `rcs_celulas_ml`, `usuario`, `razon_desviacion`, `programa_lavado`, `estado_md_pezon` | > 90 % nulos o varianza cero |
| Eliminar registros sin fecha o sin robot (`ms`) | Sin robot no se puede calcular la cola |
| Eliminar duplicados vaca + robot + hora de inicio | Integridad |
| Duración e intervalo a minutos | Necesario para los cálculos de tiempo |
| Eliminar duraciones < 0.5 min o > 30 min | Físicamente improbables |
| Eliminar el segundo registro de ordeños traslapados en el mismo robot | Dos vacas a la vez en un robot es imposible |
| `patada`, `incompleto`, `pezones_no_encotnrados` → banderas `tiene_*` | El nulo significa "no ocurrió" |
| Marcar ordeños de 0 kg como `sin_produccion` | Son reales, pero conviene poder filtrarlos |

In [ ]:
reporte = {"iniciales": sdf.count()}

# 1. Columnas sin información útil
sdf = sdf.drop("rcs_celulas_ml", "usuario", "razon_desviacion", "programa_lavado", "estado_md_pezon")

# 2. Fecha válida y robot asignado
sdf = sdf.withColumn("hora_inicio", F.to_timestamp("hora_inicio"))
sdf = sdf.filter(F.col("hora_inicio").isNotNull() & F.col("ms").isNotNull())
reporte["con_fecha_y_robot"] = sdf.count()

# 3. Duplicados
sdf = sdf.dropDuplicates(["id_vaca", "ms", "hora_inicio"])
reporte["sin_duplicados"] = sdf.count()

# 4. Duración e intervalo a minutos (acepta hh:mm:ss y mm:ss)
def a_min(col):
    t = F.element_at(F.split(F.trim(F.col(col)), " "), -1)   # '1900-01-01 00:10:44' -> '00:10:44'
    p = F.split(t, ":")
    return (F.when(F.size(p) == 3, p[0].cast("double") * 60 + p[1].cast("double") + p[2].cast("double") / 60)
             .when(F.size(p) == 2, p[0].cast("double") + p[1].cast("double") / 60))

sdf = (sdf.withColumn("duracion_min", a_min("duracion"))
          .withColumn("intervalo_min", a_min("intervalo_ordenio"))
          .drop("duracion", "intervalo_ordenio"))

# 5. Columnas numéricas
texto = {"hora_inicio", "ms", "id_vaca", "origen_archivo", "destino_leche",
         "patada", "incompleto", "pezones_no_encotnrados", "duracion_min", "intervalo_min"}
for c in sdf.columns:
    if c not in texto:
        sdf = sdf.withColumn(c, F.col(c).cast("double"))

# 6. Duraciones fuera de rango
sdf = sdf.filter(F.col("duracion_min").between(0.5, 30))
reporte["duracion_valida"] = sdf.count()

# 7. Banderas
for c in ["patada", "incompleto", "pezones_no_encotnrados"]:
    sdf = sdf.withColumn(f"tiene_{c}", F.col(c).isNotNull())
sdf = sdf.withColumn("sin_produccion", F.coalesce(F.col("produccion_kg"), F.lit(0.0)) == 0)

# 8. Tiempo libre entre ordeños del mismo robot y eliminación de traslapes
w = Window.partitionBy("ms").orderBy("hora_inicio")

def tiempo_libre(df):
    inicio = F.col("hora_inicio").cast("long")
    fin = inicio + F.col("duracion_min") * 60
    return (df.withColumn("hora_fin", fin.cast("timestamp"))
              .withColumn("tiempo_libre_min", (F.lead(inicio).over(w) - fin) / 60))

sdf = tiempo_libre(sdf)
sdf = (sdf.withColumn("_anterior", F.lag("tiempo_libre_min").over(w))
          .filter(F.col("_anterior").isNull() | (F.col("_anterior") >= 0))
          .drop("_anterior"))
sdf = tiempo_libre(sdf)
reporte["sin_traslapes"] = sdf.count()

# 9. Variable objetivo
sdf = sdf.withColumn("posible_cola",
                     F.when(F.col("tiempo_libre_min").between(0, 24 * 60),
                            F.col("tiempo_libre_min") <= UMBRAL_COLA_MIN))
sdf = sdf.withColumn("entro_en_cola", F.lag("posible_cola").over(w))

# 10. Variables temporales (dia_semana: 0 = lunes, igual que pandas)
sdf = (sdf.withColumn("fecha", F.to_date("hora_inicio"))
          .withColumn("hora", F.hour("hora_inicio"))
          .withColumn("dia_semana", (F.dayofweek("hora_inicio") + 5) % 7)
          .withColumn("mes", F.date_format("hora_inicio", "yyyy-MM"))
          .withColumn("fin_de_semana", F.col("dia_semana") >= 5))

sdf = sdf.cache()
reporte["finales"] = sdf.count()

for k, v in reporte.items():
    print(f"{k:>20}: {v:,}")
pct = sdf.agg(F.avg(F.col("posible_cola").cast("double"))).first()[0] * 100
print(f"{'% posible cola':>20}: {pct:.1f} %")

## 4. Carga
Los datos limpios se guardan en formato **Parquet** en `Datos/Procesado`. Parquet es columnar y comprimido, conserva los tipos de dato y Spark lo lee de forma nativa. Para verificar la carga se vuelven a leer con Spark y se compara el número de registros y columnas.

In [ ]:
RUTA_SALIDA = f"{RUTA_PROCESADO}/produccion_limpia"
sdf.coalesce(1).write.mode("overwrite").parquet(RUTA_SALIDA)

# Verificación
cargado = spark.read.parquet(RUTA_SALIDA)
print(f"Registros guardados: {sdf.count():,} | Registros leídos: {cargado.count():,}")
print(f"Columnas guardadas:  {len(sdf.columns)} | Columnas leídas:  {len(cargado.columns)}")
assert cargado.count() == sdf.count() and len(cargado.columns) == len(sdf.columns)
print("Carga verificada correctamente")
cargado.printSchema()

## 5. Muestra para GitHub (100 registros)

In [ ]:
muestra = cargado.orderBy(F.rand(42)).limit(100).toPandas()
muestra.to_csv(f"{RUTA_PROCESADO}/produccion_muestra.csv", index=False)
print(f"Muestra guardada en {RUTA_PROCESADO}/produccion_muestra.csv")
muestra.head()

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F

Y = "label"

dataset_ml = cargado.withColumn("label", F.col("posible_cola").cast("boolean"))
display(dataset_ml.limit(5).toPandas().head(5))



In [ ]:
# Seleccionamos variables predictoras numéricas
columnas_excluir = {
    "posible_cola", "label", "hora_inicio", "hora_fin", "fecha", "mes",
    "id_vaca", "origen_archivo", "destino_leche", "ms",
    "patada", "incompleto", "pezones_no_encotnrados"
}

features_cols = [c for c, t in dataset_ml.dtypes if c not in columnas_excluir and t in ("double", "int", "bigint", "float", "boolean")]
print(features_cols)

In [ ]:
# 2. Preparar datos para el framework / Vectorización de cada renglón
assembler = VectorAssembler(inputCols=features_cols, outputCol="features", handleInvalid="skip")
dataset_preparado = assembler.transform(dataset_ml).select("features", "label")
dataset_preparado.show(5, truncate=False)

In [ ]:
# 3. Dividir en Entrenamiento y Prueba
train, test = dataset_preparado.randomSplit([0.8, 0.2], seed=42)

# 4. Configurar el Modelo y la Validación Cruzada
lr = LogisticRegression(featuresCol="features", labelCol="label")
grid = ParamGridBuilder().addGrid(lr.maxIter, [10, 20]).build()
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds=3, # Definimos 3 folds para la validación cruzada
    parallelism=2
)

# 5. Entrenar modelo
cvModel = cv.fit(train)

print(f"Número de Folds: {cvModel.getNumFolds()}")
print(f"Métrica promedio (ROC AUC): {cvModel.avgMetrics[0]:.4f}")

# 6. Evaluar el modelo en el conjunto de prueba (Test) independiente
predictions = cvModel.transform(test)
test_auc = evaluator.evaluate(predictions)

print(f"Métrica en Test (ROC AUC): {test_auc:.4f}")